# Phase 14 — Clinically-Grounded ED/ES Frame Extraction

**Why this replaces the 8 evenly-spaced frames from notebook 12:** clinicians compute EF from exactly two frames -- End-Diastole (ED, maximum LV volume) and End-Systole (ES, minimum LV volume) -- via Simpson's method. Evenly-spaced sampling has no guarantee of hitting these frames. EchoNet-Dynamic's `VolumeTracings.csv` contains the expert-traced contours for exactly the ED and ES frames of every video, so we can extract the clinically correct frames directly, with no guessing.

**How ED vs. ES is determined:** each video has traced contours at exactly two frame indices. We compute the polygon area enclosed by each traced contour (via the shoelace formula) and label the larger-area frame as ED (maximum volume) and the smaller-area frame as ES (minimum volume) -- robust regardless of any assumption about row ordering in the CSV.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
VIDEOS_DIR = os.path.join(DATA_DIR, 'Videos')
TRACINGS_CSV = os.path.join(DATA_DIR, 'VolumeTracings.csv')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
EDES_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/edes', 'frames')
os.makedirs(EDES_CACHE_DIR, exist_ok=True)

assert os.path.exists(TRACINGS_CSV), f'VolumeTracings.csv not found at {TRACINGS_CSV}'
tracings = pd.read_csv(TRACINGS_CSV)
print('VolumeTracings.csv columns:', tracings.columns.tolist())
print(tracings.head())

## Compute ED/ES frame index per video from traced contour area

In [ ]:
def polygon_area(xs, ys):
    # Shoelace formula
    xs, ys = np.array(xs), np.array(ys)
    return 0.5 * abs(np.dot(xs, np.roll(ys, 1)) - np.dot(ys, np.roll(xs, 1)))

file_col = 'FileName' if 'FileName' in tracings.columns else [c for c in tracings.columns if 'file' in c.lower()][0]
frame_col = 'Frame' if 'Frame' in tracings.columns else [c for c in tracings.columns if 'frame' in c.lower()][0]
x_cols = [c for c in tracings.columns if c.upper() in ('X1', 'X2')]
y_cols = [c for c in tracings.columns if c.upper() in ('Y1', 'Y2')]
print('Using columns -- file:', file_col, ' frame:', frame_col, ' x:', x_cols, ' y:', y_cols)

ed_es_rows = []
for fname, group in tracings.groupby(file_col):
    frame_areas = {}
    for frame_idx, fgroup in group.groupby(frame_col):
        # Each row of a traced frame is one chord/segment (X1,Y1)-(X2,Y2); collect all endpoints
        # into a rough contour polygon to estimate enclosed area.
        pts_x = list(fgroup[x_cols[0]]) + list(fgroup[x_cols[1]])
        pts_y = list(fgroup[y_cols[0]]) + list(fgroup[y_cols[1]])
        if len(pts_x) < 3:
            continue
        frame_areas[frame_idx] = polygon_area(pts_x, pts_y)
    if len(frame_areas) < 2:
        continue  # need exactly two traced frames to identify ED vs ES
    sorted_frames = sorted(frame_areas.items(), key=lambda kv: kv[1])
    es_frame, ed_frame = sorted_frames[0][0], sorted_frames[-1][0]  # smallest area = ES, largest = ED
    ed_es_rows.append({file_col: fname, 'ED_frame': int(ed_frame), 'ES_frame': int(es_frame)})

ed_es_df = pd.DataFrame(ed_es_rows)
print(f'Identified ED/ES frame indices for {len(ed_es_df)} videos')
ed_es_df.to_csv(os.path.join(PROJECT_ROOT, 'data/edes', 'ed_es_frames.csv'), index=False)
ed_es_df.head()

## Resumable extraction: save (ED frame, ES frame) as a single 2-frame stacked array per video

Storage note: 2 frames x 10,030 videos x ~37KB is roughly 740MB -- much smaller than the earlier 8-frame extraction, so storage should not be a concern here.

In [ ]:
!pip install --quiet opencv-python-headless
import cv2, time

def get_specific_frames(video_path, frame_indices):
    cap = cv2.VideoCapture(video_path)
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            return None
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames, axis=0)  # (2, H, W, 3): [ED, ES]

start_time = time.time()
cached, missing, skipped = 0, 0, 0
for i, row in ed_es_df.iterrows():
    fname = str(row[file_col])
    if not fname.endswith('.avi'):
        fname = fname + '.avi'
    cache_path = os.path.join(EDES_CACHE_DIR, fname.replace('.avi', '.npy'))
    if os.path.exists(cache_path):
        skipped += 1
        continue
    video_path = os.path.join(VIDEOS_DIR, fname)
    if not os.path.exists(video_path):
        missing += 1
        continue
    frames = get_specific_frames(video_path, [row['ED_frame'], row['ES_frame']])
    if frames is not None:
        np.save(cache_path, frames)
        cached += 1
    else:
        missing += 1
    if (cached + skipped) % 500 == 0:
        elapsed_min = (time.time() - start_time) / 60
        print(f'[{i+1}/{len(ed_es_df)}] newly cached={cached}, already done={skipped}, missing={missing}, elapsed={elapsed_min:.1f} min')

print(f'DONE this run. newly cached={cached}, already done={skipped}, missing={missing}')
print('Safe to re-run this cell if interrupted -- completed videos are skipped.')

## Verify

In [ ]:
import glob
n_cached = len(glob.glob(os.path.join(EDES_CACHE_DIR, '*.npy')))
print(f'Total videos with cached ED/ES frames: {n_cached} / {len(ed_es_df)}')

sample = glob.glob(os.path.join(EDES_CACHE_DIR, '*.npy'))[0]
arr = np.load(sample)
print('Sample shape:', arr.shape, arr.dtype, '-- should be (2, 112, 112, 3)')

if n_cached < len(ed_es_df) * 0.98:
    print('Extraction not yet complete -- re-run the extraction cell above.')
else:
    print('Extraction complete. Proceed to notebook 15 for the ED/ES-based model.')